In [ ]:
!pip install datasets tiktoken torch -q

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
import tiktoken
import time
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")


Using: cuda


In [ ]:
# Model architecture
block_size = 256      # how many tokens the model sees at once
n_embd     = 384      # embedding dimension
n_head     = 6        # number of attention heads
n_layer    = 6        # number of transformer blocks
dropout    = 0.1

# Training
batch_size    = 32
max_iters     = 5000
eval_interval = 500
eval_iters    = 50
learning_rate = 3e-4


In [ ]:
print("Loading Python code dataset...")
dataset = load_dataset("flytech/python-codes-25k", split="train")

# Use first 8k examples for a solid training corpus
raw_text = "\n\n".join(dataset["output"][:8000])
print(f"Total characters: {len(raw_text):,}")


Loading Python code dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Total characters: 1,962,486


In [ ]:
# Using GPT-2's tokenizer (tiktoken) — same one OpenAI used
enc = tiktoken.get_encoding("gpt2")

tokens    = enc.encode(raw_text)
vocab_size = enc.n_vocab  # 50,257 tokens

print(f"Total tokens: {len(tokens):,}")
print(f"Vocab size:   {vocab_size:,}")

data = torch.tensor(tokens, dtype=torch.long)

# 90/10 train-val split
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]


Total tokens: 775,120
Vocab size:   50,257


In [ ]:
def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size]   for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)



In [ ]:
class Head(nn.Module):
    """One causal self-attention head."""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # scaled dot-product attention
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v   = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """n_head attention heads running in parallel, then projected."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Two-layer MLP with GELU activation."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Self-attention + feed-forward with residual connections & layer norm."""
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa  = MultiHeadAttention(n_head, head_size)
        self.ff  = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # residual connection
        x = x + self.ff(self.ln2(x))  # residual connection
        return x


class miniCodeGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb   = nn.Embedding(block_size, n_embd)
        self.blocks    = nn.Sequential(*[TransformerBlock() for _ in range(n_layer)])
        self.ln_f      = nn.LayerNorm(n_embd)
        self.lm_head   = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok    = self.token_emb(idx)
        pos    = self.pos_emb(torch.arange(T, device=device))
        x      = tok + pos
        x      = self.blocks(x)
        x      = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond      = idx[:, -block_size:]
            logits, _     = self(idx_cond)
            logits        = logits[:, -1, :] / temperature
            # top-k filtering
            v, _          = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
            probs         = F.softmax(logits, dim=-1)
            next_tok      = torch.multinomial(probs, num_samples=1)
            idx           = torch.cat([idx, next_tok], dim=1)
        return idx



In [ ]:
model  = miniCodeGPT().to(device)
params = sum(p.numel() for p in model.parameters())
print(f"miniCodeGPT ready — {params/1e6:.1f}M parameters")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


miniCodeGPT ready — 49.4M parameters


In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = [model(*get_batch(split))[1].item() for _ in range(eval_iters)]
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


In [ ]:
print("Training started...\n")
t0 = time.time()

for step in range(max_iters + 1):
    if step % eval_interval == 0:
        losses  = estimate_loss()
        elapsed = time.time() - t0
        print(f"step {step:5d} | train loss {losses['train']:.4f} | "
              f"val loss {losses['val']:.4f} | {elapsed:.0f}s elapsed")

    if step == max_iters:
        break

    X, Y      = get_batch('train')
    logits, loss = model(X, Y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print("\nTraining complete!")


Training started...

step     0 | train loss 10.9180 | val loss 10.9242 | 19s elapsed
step   500 | train loss 1.9939 | val loss 2.5320 | 357s elapsed
step  1000 | train loss 1.2497 | val loss 2.2497 | 701s elapsed
step  1500 | train loss 0.7998 | val loss 2.2006 | 1045s elapsed
step  2000 | train loss 0.5326 | val loss 2.3113 | 1389s elapsed
step  2500 | train loss 0.3642 | val loss 2.5333 | 1732s elapsed
step  3000 | train loss 0.2632 | val loss 2.5090 | 2077s elapsed
step  3500 | train loss 0.2091 | val loss 2.7028 | 2421s elapsed
step  4000 | train loss 0.1717 | val loss 2.8090 | 2765s elapsed
step  4500 | train loss 0.1480 | val loss 2.9308 | 3110s elapsed
step  5000 | train loss 0.1342 | val loss 2.9682 | 3454s elapsed

Training complete!


In [ ]:
torch.save(model.state_dict(), "minicodegpt.pth")
print("Saved: minicodegpt.pth")

Saved: minicodegpt.pth


In [ ]:
def complete_code(prompt: str, max_new_tokens: int = 150, temperature: float = 0.8) -> str:
    model.eval()
    tokens = enc.encode(prompt)
    idx    = torch.tensor([tokens], dtype=torch.long, device=device)
    out    = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature)
    return enc.decode(out[0].tolist())


# Test it
prompts = [
    "def fibonacci(n):",
    "def binary_search(arr, target):",
    "class Stack:",
]

for p in prompts:
    print(f"\n{'─'*50}")
    print(f"PROMPT: {p}")
    print(f"OUTPUT:\n{complete_code(p)}")



──────────────────────────────────────────────────
PROMPT: def fibonacci(n):
OUTPUT:
def fibonacci(n):
    if n<0:
        print("Incorrect input")
    elif n==1:
        return 0
    elif n==2:
        return 1
    else:
        return Fibonacci(n-1)+Fibonacci(n-2)
```

```python
def sum_two_chars(string, n):
    """Computes the sum of the items of a given list"""
    sum = 0
    for num in int:


──────────────────────────────────────────────────
PROMPT: def binary_search(arr, target):
OUTPUT:
def binary_search(arr, target):
    start = 0
    while low <= high:
        mid = (low + high)//2

        if arr[mid] == target:
            return mid

        else:
            low = mid + 1

    return -1
```

```python
import re 

def remove_special_chars(s): 
    return re.sub(r'[^\w\s]','', s) 
```



──────────────────────────────────────────────────
PROMPT: class Stack:
OUTPUT:
class Stack: 
    def __init__(self): 
        self.items = []

    def add(self, item): 
        i = 0
   